# Aprendizagem Computacional

## Applying Machine Learning to WNBA

For the 10 years, data from players, teams, coaches, games and several other metrics were gathered and arranged on this dataset. The *goal* is to use this data to predict for the test season:

- The ranking of the regular season for each conference
- Set of teams that will change coaches
- Who won each of the individual awards

![alt image](../misc/data_diagram.png)

## Datasets Description

The data about the players, teams and coaches consist of following relations:

- relation awards_players (96 objects) - each record describes awards and prizes received by players across 10 seasons,
- relation coaches (163 objects) - each record describes all coaches who've managed the teams during the time period,
- relation players (894 objects) - each record contains details of all players,
- relation players_teams (1877 objects) - each record describes the performance of each player for each team they played,
- relation series_post (71 objects) - each record describes the series' results,
- relation teams (143 objects) - each record describes the performance of the teams for each season,
- relation teams_post (81 objects) - each record describes the results of each team at the post-season.

**Offensive Statistics** (prefixed with 'o_'):
- o_fgm: Field goals made
- o_fga: Field goals attempted
- o_ftm: Free throws made
- o_fta: Free throws attempted
- o_3pm: Three-pointers made
- o_3pa: Three-pointers attempted
- o_oreb: Offensive rebounds
- o_dreb: Defensive rebounds
- o_reb: Total rebounds
- o_asts: Assists
- o_pf: Personal fouls
- o_stl: Steals
- o_to: Turnovers
- o_blk: Blocks
- o_pts: Points scored

**Defensive Statistics** (prefixed with 'd_'):
- d_fgm: Field goals made by opponents
- d_fga: Field goals attempted by opponents
- d_ftm: Free throws made by opponents
- d_fta: Free throws attempted by opponents
- d_3pm: Three-pointers made by opponents
- d_3pa: Three-pointers attempted by opponents
- d_oreb: Offensive rebounds by opponents
- d_dreb: Defensive rebounds by opponents
- d_reb: Total rebounds by opponents
- d_asts: Assists by opponents
- d_pf: Personal fouls by opponents
- d_stl: Steals by opponents
- d_to: Turnovers by opponents
- d_blk: Blocks by opponents
- d_pts: Points scored by opponents

**Team Rebounding**:
- tmORB: Team offensive rebounds
- tmDRB: Team defensive rebounds
- tmTRB: Team total rebounds

**Opponent Team Rebounding**:
- opptmORB: Opponent team offensive rebounds
- opptmDRB: Opponent team defensive rebounds
- opptmTRB: Opponent team total rebounds

**Season Performance**:
- won: Games won in the season
- lost: Games lost in the season
- GP: Games played
- homeW: Home games won
- homeL: Home games lost
- awayW: Away games won
- awayL: Away games lost
- confW: Conference games won
- confL: Conference games lost

## Business Understanding (BU)

### Analysis of Requirements

**Potential Stakeholders**: WNBA Management, Team Executives, Sports Analysts and Commentators, Fans

**Requirements**:
1. **Season Predictions**: Ability to forecast regular season standings for strategic planning
2. **Coaching Decisions**: Identify teams likely to change coaches for management insights
3. **Awards Prediction**: Anticipate individual award winners for marketing and fan engagement

**User Needs**:
- Accurate predictions to inform strategic decisions (team management, roster changes)
- Insights into performance metrics that drive success
- Understanding of factors influencing coaching stability
- Early identification of award-worthy players

**Data Availability**: 10 years of comprehensive WNBA data including player statistics, team performance, coaching records, and awards history

### Business Goals

**Primary Goals**:
1. **Improve Team Performance Prediction**: Develop models to accurately predict conference rankings
   - Success Metric: Correctly rank teams within ±2 positions of actual final standings
   - Business Value: Enables better strategic planning for playoffs and roster management

2. **Optimize Coaching Decisions**: Predict which teams are likely to change coaches
   - Success Metric: Identify 70%+ of teams that will undergo coaching changes
   - Business Value: Reduces costs of poor coaching fits and improves team stability

3. **Identify Award Winners Early**: Predict individual award recipients
   - Success Metric: Correctly predict 60%+ of major award winners
   - Business Value: Enhances marketing campaigns and fan engagement throughout season

**Secondary Goals**:
- Understand key performance indicators that differentiate successful teams
- Identify player attributes that correlate with award recognition
- Analyze impact of coaching experience on team performance

### Translation of Business Goals into Data Mining Goals

#### Goal 1: Conference Ranking Prediction
**Data Mining Task**: Regression/Ranking Problem
- **Target Variable**: Team rank within conference (1-6 or 1-7)
- **Features**: 
  - Team statistics: offensive/defensive efficiency, win%, home/away performance
  - Player statistics: NBA_PER, assist-to-turnover ratio, shooting percentages
  - Historical performance: previous season rankings, playoff appearances
- **Approach**: Ordinal regression or learning-to-rank algorithms
- **Evaluation Metric**: Mean Absolute Error (MAE) in ranking position, Spearman correlation

#### Goal 2: Coaching Change Prediction
**Data Mining Task**: Binary Classification Problem
- **Target Variable**: Coach change (Yes/No) in upcoming season
- **Features**:
  - Team performance: win%, playoff participation, win_pct variance
  - Coach statistics: experience (seasons coached), historical win%, recent performance trend
  - Situational factors: consecutive losing seasons, missing playoffs
- **Approach**: Logistic Regression, Random Forest, XGBoost
- **Evaluation Metric**: Precision, Recall, F1-Score (class imbalance likely)

#### Goal 3: Award Winner Prediction
**Data Mining Task**: Multi-class Classification Problem (per award category)
- **Target Variables**: 
  - MVP
  - Rookie of the Year
  - Defensive Player of the Year
  - Most Improved Player
  - Coach of the Year
- **Features**:
  - Player statistics: NBA_PER, ppg, apg, rpg, spg, bpg, True Shooting %
  - Team context: team win%, playoff success
  - Player position and role
  - Historical award patterns by position
- **Approach**: Multi-class classification (Random Forest, Neural Networks)
- **Evaluation Metric**: Top-3 Accuracy, Precision@K

**Data Preparation Requirements**:
- Handle missing values in player height/weight
- Feature engineering: create efficiency metrics, performance trends
- Temporal split: train on years 1-9, validate on year 10
- Address class imbalance for coaching changes and awards

**Expected Challenges**:
- Limited data (10 seasons) for deep learning approaches
- High variance in individual performance year-over-year
- Class imbalance in award winners (few winners, many candidates)
- Coaching changes may depend on external factors not in dataset


### Player Statistics

**NBA Player Efficiency Rating** = (o_pts + o_reb + o_asts + o_stl + o_blk − ((o_fga − o_fgm) + (o_fta − o_ftm) + o_to))

**Assist to Turnover Ratio** = o_asts / o_to

***IMPORTANTE*** 

[John Hollinger's Player Efficiency Rating (PER)](https://en.wikipedia.org/wiki/Player_efficiency_rating#Calculation) -> cálculo mais complexo mas permite até *prever* prémios de jogadores.

**Turnover Ratio** = o_to / GP

**Average Height / WinRatio**

## Data Analysis

### Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.ensemble import IsolationForest
from IPython.display import display

In [ ]:
sns.set_theme(style="whitegrid", palette="pastel")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
awards_players_df = pd.read_csv('../datasets/awards_players.csv')
coaches_df        = pd.read_csv('../datasets/coaches.csv')
players_teams_df  = pd.read_csv('../datasets/players_teams.csv')
players_df        = pd.read_csv('../datasets/players.csv')
series_post_df    = pd.read_csv('../datasets/series_post.csv')
teams_post_df     = pd.read_csv('../datasets/teams_post.csv')
teams_df          = pd.read_csv('../datasets/teams.csv')

### Basic Analysis

In [ ]:
print("=== Head ===")
display(awards_players_df.head())
print("\n=== Describe ===")
display(awards_players_df.describe())
print("\n=== Info ===")
awards_players_df.info()
print("\n=== Missing Values per Column ===")
display(awards_players_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {awards_players_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(awards_players_df.nunique())

The lgID column is redundant, as it only contains 1 value ("WNBA"). Therefore, it can be safely removed.

In [ ]:
print("=== Head ===")
display(coaches_df.head())
print("\n=== Describe ===")
display(coaches_df.describe())
print("\n=== Info ===")
coaches_df.info()
print("\n=== Missing Values per Column ===")
display(coaches_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {coaches_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(coaches_df.nunique())

Just like in the previous dataframe, the lgID column is redundant.

In [ ]:
print("=== Head ===")
display(players_teams_df.head())
print("\n=== Describe ===")
display(players_teams_df.describe())
print("\n=== Info ===")
players_teams_df.info()
print("\n=== Missing Values per Column ===")
display(players_teams_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {players_teams_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(players_teams_df.nunique())

Once again, the lgID column is redundant.

In [ ]:
print("=== Head ===")
display(players_df.head())
print("\n=== Describe ===")
display(players_df.describe())
print("\n=== Info ===")
players_df.info()
print("\n=== Missing Values per Column ===")
display(players_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {players_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(players_df.nunique())

- **Weight** and **height** columns have values of 0, which should be impossible;
- **first** and **last** season columns can be removed since values are the same for every player;
- There are 78 entries where the values for the **pos** column are missing, and several entries with a null birth date ("0000-00-00").

In [ ]:
print("=== Head ===")
display(series_post_df.head())
print("\n=== Describe ===")
display(series_post_df.describe())
print("\n=== Info ===")
series_post_df.info()
print("\n=== Missing Values per Column ===")
display(series_post_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {series_post_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(series_post_df.nunique())

The lgID columns (lgIDWinner and lgIDLoser) are redundant.

In [ ]:
print("=== Head ===")
display(teams_post_df.head())
print("\n=== Describe ===")
display(teams_post_df.describe())
print("\n=== Info ===")
teams_post_df.info()
print("\n=== Missing Values per Column ===")
display(teams_post_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {teams_post_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(teams_post_df.nunique())

The lgID column is redundant.

In [ ]:
print("=== Head ===")
display(teams_df.head())
print("\n=== Describe ===")
display(teams_df.describe())
print("\n=== Info ===")
teams_df.info()
print("\n=== Missing Values per Column ===")
display(teams_df.isnull().sum())
print("\n=== Duplicate Rows ===")
print(f"Number of duplicate rows: {teams_df.duplicated().sum()}")
print("\n=== Unique Values per Column ===")
display(teams_df.nunique())

The lgID, seeded, tmORB, tmDRB, tmTRB, opptmORB, opptmDRB, opptmTRB columns are redundant and can be removed. The divID column can be removed as well as all of its values are missing. Missing values in other columns should be properly handled.

## Coach Analysis

In [ ]:
# Calculate win percentage for each coach
coaches_df['win_pct'] = coaches_df['won'] / (coaches_df['won'] + coaches_df['lost'])
top_coaches = coaches_df.groupby('coachID')['win_pct'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
top_coaches.plot(kind='bar')
plt.title('Top 10 Coaches by Win Percentage')
plt.ylabel('Win Percentage')
plt.xlabel('Coach ID')
plt.show()

In [ ]:
# Coaches with more experience (num of seasons coached)
coach_seasons = coaches_df.groupby('coachID')['year'].nunique().sort_values(ascending=False).head(10)
coach_seasons.plot(kind='bar', color='orange')
plt.title('Top 10 Coaches by Experience')
plt.xlabel('Coach ID')
plt.ylabel('Seasons')
plt.show()

In [ ]:
# Distribution of wins and losses for top coaches
top_coach_ids = top_coaches.index.tolist()
top_coach_stats = coaches_df[coaches_df['coachID'].isin(top_coach_ids)].groupby('coachID')[['won', 'lost']].sum()

top_coach_stats.plot(kind='bar', stacked=True)

plt.title('Wins vs Losses for Top 10 Coaches')
plt.xlabel('Coach ID')
plt.ylabel('Games')
plt.legend(['Wins', 'Losses'])
plt.show()

In [ ]:
# Coach experience vs. average win percentage
coach_experience = coaches_df.groupby('coachID')['year'].nunique()
coach_win_pct = coaches_df.groupby('coachID').apply(lambda x: x['won'].sum() / (x['won'].sum() + x['lost'].sum()))

plt.scatter(coach_experience, coach_win_pct, alpha=0.7)
plt.title('Coach Experience vs. Average Win Percentage')
plt.xlabel('Number of Seasons Coached')
plt.ylabel('Average Win Percentage')
plt.show()

Coaches with more seasons tend to cluster around the middle win percentage (about 0.5–0.6). Coaches with only 1–2 seasons have a wide range of win percentages.

In [ ]:
# Wins and Losses
coaches_df["total_w"] = coaches_df["won"] + coaches_df["post_wins"]
coaches_df["total_l"] = coaches_df["lost"] + coaches_df["post_losses"]

## Player Analysis

In [ ]:
# Distribution of player heights
plt.figure(figsize=(10, 6))
players_df['height'].hist(bins=30, color='green', alpha=0.7)
plt.title('Distribution of Player Heights (Inches)')
plt.xlabel('Height (Inches)')
plt.ylabel('Number of Players')
plt.grid(False)
plt.show()

This graph shows that data cleaning is needed: there are several players with the height recorded as 0.

In [ ]:
# Distribution of player weights
plt.figure(figsize=(10, 6))
players_df['weight'].hist(bins=30, color='purple', alpha=0.7)
plt.title('Distribution of Player Weights (lbs)')
plt.xlabel('Weight (lbs)')
plt.ylabel('Number of Players')
plt.grid(False)
plt.show()

The same happens with the weight.

In [ ]:
# Distribution of player career lengths
player_career_lengths = players_teams_df.groupby('playerID')['year'].nunique()
plt.figure(figsize=(10, 6))
player_career_lengths.hist(bins=30, color='orange', alpha=0.7)
plt.title('Distribution of Player Career Lengths (Seasons)')
plt.xlabel('Number of Seasons Played')
plt.ylabel('Number of Players')
plt.grid(False)
plt.show()

In [ ]:
# Top 10 players with the most awards
top_award_players = awards_players_df['playerID'].value_counts().head(10)
top_award_players.plot(kind='bar', color='gold')
plt.title('Top 10 Players by Number of Awards')
plt.xlabel('Player ID')
plt.ylabel('Number of Awards')
plt.show()

In [ ]:
# Distribution of award types
award_types = awards_players_df['award'].value_counts()
plt.figure(figsize=(12, 7))
award_types.plot(kind='barh', color='silver')
plt.title('Distribution of Award Types')
plt.xlabel('Number of Times Awarded')
plt.ylabel('Award')
plt.show()

In [ ]:
# Calculate per-game averages for key stats
player_stats = players_teams_df.copy()
player_stats['ppg'] = player_stats['points'] / player_stats['GP']
player_stats['apg'] = player_stats['assists'] / player_stats['GP']
player_stats['rpg'] = player_stats['rebounds'] / player_stats['GP']
player_stats['spg'] = player_stats['steals'] / player_stats['GP']
player_stats['bpg'] = player_stats['blocks'] / player_stats['GP']

# Top 10 players by points per game (min 10 games played)
ppg_leaders = player_stats[player_stats['GP'] >= 10].groupby('playerID')['ppg'].mean().sort_values(ascending=False).head(10)
ppg_leaders.plot(kind='bar', color='crimson')
plt.title('Top 10 Players by Points Per Game')
plt.xlabel('Player ID')
plt.ylabel('Points Per Game')
plt.show()

In [ ]:
# Boxplot: Points per game by position
merged = pd.merge(player_stats, players_df[['bioID','pos']], left_on='playerID', right_on='bioID', how='left')
boxplot_data = merged[['pos', 'ppg']].dropna()
sns.boxplot(x='pos', y='ppg', data=boxplot_data)
plt.title('Points Per Game by Position')
plt.xlabel('Position')
plt.ylabel('Points Per Game')
plt.show()

Players in forward positions are generally more productive scorers, as expected.

In [ ]:
# Heatmap: Correlation between advanced stats
corr = player_stats[['ppg','apg','rpg','spg','bpg']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Per-Game Stats')
plt.show()

Points (ppg) and rebounds (rpg) are highly correlated: players who score more also tend to rebound more. Points and assists (apg) are moderately correlated, suggesting that high scorers also often assist more. There's a fairly strong correlation between assists and steals (spg), indicating that playmakers who frequently assist are also active defensively. Blocks (bpg) are weakly correlated with assists and steals, but more with rebounds.

In [ ]:
# Violin plot: Distribution of field goal % by position
player_stats['fg_pct'] = player_stats['fgMade'] / player_stats['fgAttempted']
merged = pd.merge(player_stats, players_df[['bioID','pos']], left_on='playerID', right_on='bioID', how='left')
sns.violinplot(x='pos', y='fg_pct', data=merged)
plt.title('Field Goal Percentage by Position')
plt.xlabel('Position')
plt.ylabel('Field Goal %')
plt.show()

Position affects shooting efficiency: centers and forwards are more efficient, likely due to closer shots.

In [ ]:
# Radar chart: Average stat profile for each position (all positions together)
import matplotlib.pyplot as plt
import numpy as np

positions = merged['pos'].dropna().unique()
stat_cols = ['ppg','apg','rpg','spg','bpg']
angles = np.linspace(0, 2 * np.pi, len(stat_cols), endpoint=False).tolist()
angles += angles[:1]

# Use a more distinguishable color palette
colors = plt.get_cmap('tab10').colors  # tab10 has 10 distinct colors

plt.figure(figsize=(8,8))
for i, pos in enumerate(positions):
    values = merged[merged['pos']==pos][stat_cols].mean().tolist()
    values += values[:1]
    plt.polar(angles, values, label=pos, color=colors[i % len(colors)])  # assign color

plt.xticks(angles[:-1], stat_cols)
plt.title('Average Stat Profile by Position (Radar Chart)')
plt.legend()
plt.show()

This radar chart shows that each position has a distinct statistical profile, namely:

- Centers: rebounding and blocking
- Guards: playmaking and steals
- Forwards: scoring

In [ ]:
# Pie chart: Proportion of players by position
pos_counts = players_df['pos'].value_counts()
plt.figure(figsize=(7,7))
plt.pie(pos_counts, labels=pos_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Proportion of Players by Position')
plt.show()

There is a relevant class imbalance in the sense that hybrid positions are much less frequent in the dataset.

In [ ]:
# Win Percentage
teams_df['win_pct'] = (teams_df['won'] / teams_df['GP']) * 100

merged = pd.merge(players_teams_df, teams_df[["year", "tmID", "win_pct"]], on=["year", "tmID"], how="left")

merged = merged.drop_duplicates(subset=["playerID"], keep="first")

merged = pd.merge(merged, players_df[["bioID", "height"]], left_on="playerID", right_on='bioID', how="left")

# inches to cm
merged['height'] *= 2.54

bins = [0, 170, 175, 180, 185, 190, 195, 200, 230]

labels = [
    '< 170cm', '170-174cm', '175-179cm', '180-184cm',
    '185-189cm', '190-194cm', '195-200cm', '> 200cm'
]

merged['height_bin'] = pd.cut(merged['height'], bins=bins, labels=labels, right=False)

plt.figure(figsize=(10, 6))
plt.title('Relationship between player height and team win%')
sns.barplot(data=merged, x='height_bin', y='win_pct', hue='height_bin', palette='Blues')
plt.show()


Average player height is not a strong predictor of team success.

In [ ]:
# Assist to turnover Ratio
p = players_teams_df.copy()

p["ast_to"] = (p["assists"] + p["PostAssists"]) / (p["turnovers"] + p["PostTurnovers"])

plt.figure(figsize=(10, 8))
sns.histplot(p['ast_to'], bins=30, kde=True, color='skyblue')

plt.axvline(3, color='red', linestyle='--', label='3:1 Ideal')

plt.title('Distribution of Assist-to-Turnover Ratios')

plt.xlabel('Assist-to-Turnover Ratio')
plt.ylabel('Number of Players')

plt.legend()
plt.show()


In [ ]:
# NBA Player Efficiency Rating
# (PTS + REB + AST + STL + BLK − ((FGA − FGM) + (FTA − FTM) + TO)) / GP

players_teams_df["NBA_PER"] = (players_teams_df['points'] + players_teams_df['rebounds']+ players_teams_df['assists'] + players_teams_df['steals'] + players_teams_df['blocks']
  - ((players_teams_df['fgAttempted'] - players_teams_df['fgMade']) + (players_teams_df['ftAttempted'] - players_teams_df['ftMade']) + players_teams_df['turnovers'])
) / players_teams_df["GP"]

top_by_year = (
    players_teams_df
    .groupby('year', group_keys=False)
    .apply(lambda x: x.nlargest(1, 'NBA_PER')) # change num to increase players per year
)

top_by_year[['year', 'playerID', 'tmID', 'NBA_PER']]


In [ ]:
plt.figure(figsize=(12,8))

sns.histplot(players_teams_df['NBA_PER'], bins=30, kde=True, color='skyblue')

plt.title('Distribution of NBA Performance Ratios')

plt.xlabel('NBA Performance Ratios')
plt.ylabel('Number of Players')
plt.show()


In [ ]:
award_counts = awards_players_df.groupby(['playerID']).size().reset_index(name='num_awards') 
num_awards_df = players_teams_df.merge(award_counts, on="playerID")

plt.figure(figsize=(10, 6))

sns.boxplot(data=num_awards_df, x='num_awards', y='NBA_PER')

plt.title('Player Efficiency Rating (PER) by Number of Awards')
plt.xlabel('Number of Awards')
plt.ylabel('NBA PER')

In general, players with more awards generally have higher median PER values, suggesting this could be a good predictor of individual awards.

In [ ]:
# True Shooting Percentage
players_teams_df["ts_pct"] = players_teams_df['points'] / (2 * (players_teams_df["fgAttempted"] + (0.44 * players_teams_df["ftAttempted"])))

In [ ]:
p_awards = players_teams_df.merge(awards_players_df, on=["playerID", "year"], how="left")

# 1 if player won an award, 0 otherwise
p_awards["award_winner"] = p_awards["award"].notnull().astype(int)

stats = [
    "points", "rebounds", "assists", "steals", "blocks",
    "oRebounds", "dRebounds", "turnovers", "fgMade", "fgAttempted", "NBA_PER",
]

# drop null values
p_awards = p_awards.dropna(subset=stats)

mean_comparison = (
    p_awards.groupby("award_winner")[stats]
    .mean()
    .rename(index={0: "Non-Winners", 1: "Award Winners"})
    .T
)

mean_comparison["Win/NonWin Ratio"] = mean_comparison.apply(
    lambda row: row["Award Winners"] / row["Non-Winners"] , axis=1
)

mean_comparison = mean_comparison.round(2)

print("🏆 Average performance metrics for award winners vs non-winners:")
display(mean_comparison)

stats_plot = ["points", "assists", "rebounds", "NBA_PER"]

fig, axes = plt.subplots(1, len(stats_plot), figsize=(18, 5), facecolor='white')

for i, stat in enumerate(stats_plot):
    sns.boxplot(data=p_awards, x="award_winner", y=stat, ax=axes[i])
    axes[i].set_title(stat.title(), fontsize=12)

fig.suptitle("Award Winners vs Non-Winners — Statistical Comparison", fontsize=16, weight='bold')
fig.text(0.5, 0.04, "Award Winner (0 = No, 1 = Yes)", ha='center')
fig.text(0.04, 0.5, "Stat Value", va='center', rotation='vertical')

plt.tight_layout(rect=[0.03, 0.05, 1, 0.95])
plt.show()


As expected, award winners achieve consistently better statistics than non-winners.

In [ ]:
# Position analysis

# Merge dataframes without coach of the year (there are players that are both players and coaches)
position_awards = awards_players_df[awards_players_df["award"] != "Coach of the Year"] \
  .merge(players_df, left_on="playerID", right_on="bioID", how="left")

# Count how many times each position has won each award
position_awards = position_awards.groupby(["award", "pos"]) \
    .size() \
    .reset_index(name="count")  

position_awards["percentage"] = (
    position_awards.groupby("award")["count"]
    .transform(lambda x: 100 * x / x.sum())
)

# Display summary table
print("Positions distribution by award:")
display(position_awards.sort_values(["award", "percentage"], ascending=[True, False]))

# Visualization
plt.figure(figsize=(18,10))
sns.barplot(data=position_awards, x="award", y="percentage", hue="pos", palette="Set2")

plt.title("Position Distribution by Award")
plt.ylabel("Percentage of Winners (%)")
plt.xlabel("Award Type")

plt.xticks(rotation=30, ha="right") # x tilted
plt.legend(title="Position")
plt.show()

Certain positions are much more likely to win specific awards. For example, guards dominate awards like Sixth Woman of the Year and Sportmanship, while forwards and centers are more likely to win MVP.

In [ ]:
numeric_cols = players_teams_df.select_dtypes(include=['float64', 'int64']).columns.drop(list(players_teams_df.filter(regex="^Post.*")))

plt.figure(figsize=(20, 16))
for i, col in enumerate(numeric_cols):
    plt.subplot(6, 6, i+1)
    sns.boxplot(x=players_teams_df[col])
    plt.title(col)
    plt.tight_layout()

There are several features with a lot of outliers, such as points, assists, rebounds, etc. This could affect model training.

## Team Analysis

In [ ]:
d = teams_df.select_dtypes(['int64', 'float64']).copy()

d = d.drop(columns=['year', 'rank', 'seeded', 'divID'])

col_index = d.columns.get_loc("tmORB")
d = d.iloc[:, :col_index]

corr = d.corr()

plt.figure(figsize=(24, 16))
plt.title("Correlation of team attributes")
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.show()

In [ ]:
# Team win/loss records over time
team_wins = teams_df.groupby('year')['won'].sum()
team_losses = teams_df.groupby('year')['lost'].sum()

print(team_wins.head())

plt.plot(team_wins.index, team_wins.values, label='Wins')
plt.plot(team_losses.index, team_losses.values, label='Losses')
plt.title('Total Team Wins and Losses Over Time')
plt.xlabel('Year')
plt.ylabel('Games')
plt.legend()
plt.show()

In [ ]:
# Team win percentage over time
teams_df['win_pct'] = teams_df['won'] / (teams_df['won'] + teams_df['lost'])

# Top teams by overall win percentage
team_performance = teams_df.groupby('name').agg({
    'won': 'sum',
    'lost': 'sum',
    'year': 'count'
}).rename(columns={'year': 'seasons'})

team_performance['total_games'] = team_performance['won'] + team_performance['lost']
team_performance['win_pct'] = team_performance['won'] / team_performance['total_games']

# Filter teams with at least 3 seasons
experienced_teams = team_performance[team_performance['seasons'] >= 3].sort_values('win_pct', ascending=False)

plt.figure(figsize=(12, 8))
experienced_teams['win_pct'].head(10).plot(kind='bar', color='steelblue')
plt.title('Top 10 Teams by Win Percentage (3+ Seasons)')
plt.ylabel('Win Percentage')
plt.xlabel('Team')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Top performing teams:")
print(experienced_teams.head())

In [ ]:
# Team performance consistency over time
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Win percentage distribution
axes[0,0].hist(teams_df['win_pct'], bins=20, alpha=0.7, color='lightblue', edgecolor='black')
axes[0,0].set_title('Distribution of Team Win Percentages')
axes[0,0].set_xlabel('Win Percentage')
axes[0,0].set_ylabel('Frequency')

# Games played over time (to see if schedule changed)
games_per_year = teams_df.groupby('year')['GP'].mean()
axes[0,1].plot(games_per_year.index, games_per_year.values, marker='o', color='green')
axes[0,1].set_title('Average Games Played Per Team Over Time')
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Average Games')

# Win percentage variance for teams with multiple seasons
team_variance = teams_df.groupby('name')['win_pct'].agg(['mean', 'std', 'count'])
consistent_teams = team_variance[team_variance['count'] >= 5].sort_values('std')

axes[1,0].barh(range(len(consistent_teams.head(10))), consistent_teams.head(10)['std'], color='coral')
axes[1,0].set_yticks(range(len(consistent_teams.head(10))))
axes[1,0].set_yticklabels(consistent_teams.head(10).index, fontsize=8)
axes[1,0].set_title('Most Consistent Teams (Lowest Win % Variance)')
axes[1,0].set_xlabel('Win Percentage Standard Deviation')

# Points scored vs points allowed
axes[1,1].scatter(teams_df['o_pts'], teams_df['d_pts'], alpha=0.6, c=teams_df['win_pct'], cmap='RdYlGn')
axes[1,1].set_xlabel('Points Scored')
axes[1,1].set_ylabel('Points Allowed')
axes[1,1].set_title('Offensive vs Defensive Performance')
cbar = plt.colorbar(axes[1,1].collections[0], ax=axes[1,1])
cbar.set_label('Win Percentage')

plt.tight_layout()
plt.show()

In [ ]:
# Playoff participation analysis
playoff_teams = teams_df[teams_df['playoff'] == 'Y']
playoff_participation = teams_df.groupby('name')['playoff'].apply(lambda x: (x == 'Y').sum()).sort_values(ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Teams with most playoff appearances
playoff_participation.head(10).plot(kind='bar', ax=axes[0,0], color='gold')
axes[0,0].set_title('Teams with Most Playoff Appearances')
axes[0,0].set_ylabel('Playoff Appearances')
axes[0,0].tick_params(axis='x', rotation=45)

# Playoff success rates (finals appearances)
finals_teams = teams_df[teams_df['finals'].notna()]
finals_appearances = teams_df.groupby('name')['finals'].apply(lambda x: x.notna().sum()).sort_values(ascending=False)
finals_appearances[finals_appearances > 0].head(8).plot(kind='bar', ax=axes[0,1], color='silver')
axes[0,1].set_title('Teams with Most Finals Appearances')
axes[0,1].set_ylabel('Finals Appearances')
axes[0,1].tick_params(axis='x', rotation=45)

# Regular season vs playoff performance correlation
if not teams_post_df.empty:
    # Merge regular season and playoff data
    playoff_merged = teams_df.merge(teams_post_df, on=['year', 'tmID'], suffixes=('_reg', '_playoff'))
    if not playoff_merged.empty:
        axes[1,0].scatter(playoff_merged['win_pct'], playoff_merged['W'], alpha=0.7, color='red')
        axes[1,0].set_xlabel('Regular Season Win %')
        axes[1,0].set_ylabel('Playoff Wins')
        axes[1,0].set_title('Regular Season Success vs Playoff Wins')

# Championship winners over time
champions = teams_df[teams_df['finals'] == 'W']
if not champions.empty:
    champion_counts = champions['name'].value_counts()
    champion_counts.plot(kind='bar', ax=axes[1,1], color='darkgreen')
    axes[1,1].set_title('Championship Winners')
    axes[1,1].set_ylabel('Championships')
    axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("Playoff participation summary:")
print(f"Total seasons: {len(teams_df)}")
print(f"Playoff appearances: {len(playoff_teams)}")
print(f"Playoff participation rate: {len(playoff_teams)/len(teams_df):.1%}")

In [ ]:
# Correlation between key statistics and winning
import seaborn as sns

# Select key offensive and defensive statistics
key_stats = ['o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm', 'o_reb', 'o_asts', 'o_stl', 'o_to', 'o_pts',
             'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_reb', 'd_asts', 'd_stl', 'd_to', 'd_pts',
             'won', 'win_pct']
    
# Create correlation matrix
correlation_matrix = teams_df[key_stats].corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('Correlation Matrix: Team Statistics vs Performance')
plt.tight_layout()
plt.show()

# Show strongest correlations with winning percentage
win_correlations = correlation_matrix['win_pct'].sort_values(key=abs, ascending=False)
print("Strongest correlations with win percentage:")
print(win_correlations[1:11])  # Skip self-correlation

In [ ]:
# Offensive vs Defensive efficiency analysis
teams_df['off_efficiency'] = teams_df['o_pts'] / teams_df['o_fga']  # Points per shot attempt
teams_df['def_efficiency'] = teams_df['d_pts'] / teams_df['d_fga']  # Opponent points per their shot attempt
teams_df['turnover_ratio'] = teams_df['o_stl'] / teams_df['o_to']   # Steals vs turnovers

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Offensive efficiency vs win percentage
axes[0,0].scatter(teams_df['off_efficiency'], teams_df['win_pct'], alpha=0.6, color='blue')
axes[0,0].set_xlabel('Offensive Efficiency (Points per Shot)')
axes[0,0].set_ylabel('Win Percentage')
axes[0,0].set_title('Offensive Efficiency vs Winning')

# Defensive efficiency vs win percentage (lower is better for defense)
axes[0,1].scatter(teams_df['def_efficiency'], teams_df['win_pct'], alpha=0.6, color='red')
axes[0,1].set_xlabel('Defensive Efficiency (Opp Points per Shot)')
axes[0,1].set_ylabel('Win Percentage')
axes[0,1].set_title('Defensive Efficiency vs Winning')

# Rebounding analysis
teams_df['reb_diff'] = teams_df['o_reb'] - teams_df['d_reb']
axes[1,0].scatter(teams_df['reb_diff'], teams_df['win_pct'], alpha=0.6, color='green')
axes[1,0].set_xlabel('Rebounding Differential (Off - Def)')
axes[1,0].set_ylabel('Win Percentage')
axes[1,0].set_title('Rebounding Impact on Winning')

# Three-point shooting analysis
teams_df['three_pct'] = teams_df['o_3pm'] / teams_df['o_3pa']
teams_df['opp_three_pct'] = teams_df['d_3pm'] / teams_df['d_3pa']
axes[1,1].scatter(teams_df['three_pct'], teams_df['win_pct'], alpha=0.6, color='purple')
axes[1,1].set_xlabel('Three-Point Percentage')
axes[1,1].set_ylabel('Win Percentage')
axes[1,1].set_title('Three-Point Shooting vs Winning')

plt.tight_layout()
plt.show()

In [ ]:
# Conference performance comparison
conference_performance = teams_df.groupby('confID').agg({
    'win_pct': 'mean',
    'o_pts': 'mean',
    'd_pts': 'mean',
    'year': 'count'
}).rename(columns={'year': 'team_seasons'})

conference_performance = conference_performance[conference_performance['team_seasons'] > 10]  # Filter for significance

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Conference win percentages
conference_performance['win_pct'].plot(kind='bar', ax=axes[0,0], color='lightcoral')
axes[0,0].set_title('Average Win Percentage by Conference')
axes[0,0].set_ylabel('Win Percentage')
axes[0,0].tick_params(axis='x', rotation=0)

# Conference scoring
conference_performance[['o_pts', 'd_pts']].plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Average Points Scored/Allowed by Conference')
axes[0,1].set_ylabel('Points per Game')
axes[0,1].tick_params(axis='x', rotation=0)
axes[0,1].legend(['Points Scored', 'Points Allowed'])

# Competitive balance - win percentage distribution by conference
for i, conf in enumerate(teams_df['confID'].unique()):
    if pd.notna(conf) and conf != '':
        conf_data = teams_df[teams_df['confID'] == conf]['win_pct']
        if len(conf_data) > 5:  # Only if enough data
            axes[1,0].hist(conf_data, alpha=0.6, label=conf, bins=15)

axes[1,0].set_title('Win Percentage Distribution by Conference')
axes[1,0].set_xlabel('Win Percentage')
axes[1,0].set_ylabel('Frequency')
axes[1,0].legend()

# Playoff success by conference
conference_playoffs = teams_df[teams_df['playoff'] == 'Y'].groupby('confID').size()
conference_total = teams_df.groupby('confID').size()
playoff_rate = (conference_playoffs / conference_total).dropna()

playoff_rate.plot(kind='bar', ax=axes[1,1], color='gold')
axes[1,1].set_title('Playoff Participation Rate by Conference')
axes[1,1].set_ylabel('Playoff Rate')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("Conference performance summary:")
print(conference_performance)

In [ ]:
# Home court advantage analysis
teams_df['home_win_pct'] = teams_df['homeW'] / (teams_df['homeW'] + teams_df['homeL'])
teams_df['away_win_pct'] = teams_df['awayW'] / (teams_df['awayW'] + teams_df['awayL'])
teams_df['home_advantage'] = teams_df['home_win_pct'] - teams_df['away_win_pct']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Home vs Away win percentage scatter
axes[0,0].scatter(teams_df['home_win_pct'], teams_df['away_win_pct'], alpha=0.6, color='brown')
axes[0,0].plot([0, 1], [0, 1], 'r--', alpha=0.5)  # Diagonal line for reference
axes[0,0].set_xlabel('Home Win Percentage')
axes[0,0].set_ylabel('Away Win Percentage')
axes[0,0].set_title('Home vs Away Win Percentage')

# Home advantage distribution
axes[0,1].hist(teams_df['home_advantage'], bins=20, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0,1].axvline(teams_df['home_advantage'].mean(), color='red', linestyle='--', 
                  label=f'Mean: {teams_df["home_advantage"].mean():.3f}')
axes[0,1].set_xlabel('Home Advantage (Home Win% - Away Win%)')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Distribution of Home Court Advantage')
axes[0,1].legend()

# Home advantage over time
home_adv_by_year = teams_df.groupby('year')['home_advantage'].mean()
axes[1,0].plot(home_adv_by_year.index, home_adv_by_year.values, marker='o', color='darkblue')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Average Home Advantage')
axes[1,0].set_title('Home Court Advantage Over Time')

# Teams with strongest home advantage
top_home_teams = teams_df.groupby('name')['home_advantage'].mean().sort_values(ascending=False)
top_home_teams.head(10).plot(kind='bar', ax=axes[1,1], color='orange')
axes[1,1].set_title('Teams with Strongest Home Advantage')
axes[1,1].set_ylabel('Average Home Advantage')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"League average home court advantage: {teams_df['home_advantage'].mean():.3f}")
print(f"Home win percentage: {teams_df['home_win_pct'].mean():.3f}")
print(f"Away win percentage: {teams_df['away_win_pct'].mean():.3f}")

In [ ]:
# Attendance analysis
# Calculate attendance per game
teams_df['attendance_per_game'] = teams_df['attend'] / teams_df['GP']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Attendance vs Win Percentage
axes[0,0].scatter(teams_df['attendance_per_game'], teams_df['win_pct'], alpha=0.6, color='purple')
axes[0,0].set_xlabel('Attendance per Game')
axes[0,0].set_ylabel('Win Percentage')
axes[0,0].set_title('Attendance vs Team Performance')

# Attendance trends over time
attendance_by_year = teams_df.groupby('year')['attendance_per_game'].mean()
axes[0,1].plot(attendance_by_year.index, attendance_by_year.values, marker='o', color='darkred')
axes[0,1].set_xlabel('Year')
axes[0,1].set_ylabel('Average Attendance per Game')
axes[0,1].set_title('League Attendance Trends')

# Top teams by attendance
top_attendance = teams_df.groupby('name')['attendance_per_game'].mean().sort_values(ascending=False)
top_attendance.head(10).plot(kind='bar', ax=axes[1,0], color='teal')
axes[1,0].set_title('Teams with Highest Average Attendance')
axes[1,0].set_ylabel('Attendance per Game')
axes[1,0].tick_params(axis='x', rotation=45)

# Attendance vs Home Win Percentage
axes[1,1].scatter(teams_df['attendance_per_game'], teams_df['home_win_pct'], alpha=0.6, color='orange')
axes[1,1].set_xlabel('Attendance per Game')
axes[1,1].set_ylabel('Home Win Percentage')
axes[1,1].set_title('Attendance vs Home Performance')

plt.tight_layout()
plt.show()

# Calculate correlation between attendance and performance
attendance_correlation = teams_df[['attendance_per_game', 'win_pct', 'home_win_pct']].corr()
print("Attendance correlations:")
print(attendance_correlation['attendance_per_game'].sort_values(ascending=False))

In [ ]:
# Check columns with missing values
teams_df.isna().any()[teams_df.isna().any()]

`divID` should be removed since every value is null. The cols `firstRound`, `semis` and `finals` should have missing values since they are only filled when a team reaches playoffs.

In [ ]:
plt.figure(figsize=(10,8))

sns.scatterplot(
  data=teams_df, 
  x="o_pts", 
  y="d_pts", 
  hue=teams_df["rank"].astype("category"), 
  palette="coolwarm"
)

plt.title("Scored vs Conceded Points by Rank")
plt.xlabel("Points Scored")
plt.ylabel("Points Conceded")
plt.show()

In [ ]:
# Statistical demonstration of relation between points

print("📉 Ratio lower than 1 (more points conceded than scored)")
f = teams_df[teams_df["o_pts"] / teams_df["d_pts"] < 1]
display(f["rank"].describe())

print("📈 Ratio higher than 1 (more points scored than conceded)")
f = teams_df[teams_df["o_pts"] / teams_df["d_pts"] > 1]
display(f["rank"].describe())

**Important** analysis since this clearly separates teams -> teams which have higher score/conceded ratio tend to rank higher in the regular season rankings. 

In [ ]:
corr = teams_df.corr(numeric_only=True)
corr_rank = corr["rank"].sort_values()

corr_rank.plot(kind='barh', figsize=(10,12))
plt.title("Top Features Correlated with Rank")
plt.xlabel("Correlation with Rank")
plt.show()

In [ ]:
corr_winpct = corr["win_pct"].sort_values(ascending=False)

corr_winpct.head(15).plot(kind='barh', figsize=(6,6))
plt.title("Top Features Correlated with Win%")
plt.show()

In [ ]:
# Helper to understand year rankings
teams_df[teams_df["year"] == 9].sort_values(by="rank")[["name","confID","win_pct", "rank", "playoff"]]

In [ ]:
# Champions by year
print("🏆 Champions per year and corresponding win%")
display(teams_df[teams_df["finals"] == "W"].sort_values(by="year")[["year","name", "confID", "win_pct"]])

[ ] **Note:** Although there are 2 winners in year 6, this probably won't affect the prediction since we want to predict the ranking, not the WNBA season winner. 

In [ ]:
# Columns without values
zero_cols = (teams_df == 0).all()
zero_cols = zero_cols[zero_cols].index.tolist()

display(zero_cols)

In [ ]:
mean_rank = teams_df.groupby("playoff")["rank"].describe()

print(mean_rank)

In [ ]:
teams_per = ( 
  players_teams_df.groupby(["year", "tmID"])["NBA_PER"] 
  .mean()
  .reset_index(name="avg_team_PER")
)

teams_per = teams_df.merge(teams_per, on=["year", "tmID"], how="left")

plt.figure(figsize=(12, 8))

sns.boxplot(
    data=teams_per,
    x="rank",           # Team rank
    y="avg_team_PER",   # Average PER
)

plt.title("Distribution of Average Team PER by Ranking")
plt.xlabel("Rank")
plt.ylabel("Average Team PER")
plt.show()

# Data Preprocessing

Read the CSV files again to clear any modifications in EDA phase.

In [ ]:
awards_players_df = pd.read_csv('../datasets/awards_players.csv')
coaches_df        = pd.read_csv('../datasets/coaches.csv')
players_teams_df  = pd.read_csv('../datasets/players_teams.csv')
players_df        = pd.read_csv('../datasets/players.csv')
series_post_df    = pd.read_csv('../datasets/series_post.csv')
teams_post_df     = pd.read_csv('../datasets/teams_post.csv')
teams_df          = pd.read_csv('../datasets/teams.csv')

### Cleaning - Missing and Redundant Values

The following variables are redundant or irrelevant, therefore they can be safely removed.

In [ ]:
coaches_df.drop(columns=['lgID'], inplace=True)
awards_players_df.drop(columns=['lgID'], inplace=True)
players_teams_df.drop(columns=['lgID'], inplace=True)
players_df.drop(columns=['firstseason', 'lastseason', 'deathDate'], inplace=True)
series_post_df.drop(columns=['lgIDWinner', 'lgIDLoser'], inplace=True)
teams_post_df.drop(columns=['lgID'], inplace=True)
teams_df.drop(columns=['lgID', 'divID', 'seeded', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB'], inplace=True)

In [ ]:
# Handle missing values in players dataframe

print("Current position distribution:")
display(players_df['pos'].value_counts())
print(f"\nNull positions: {players_df['pos'].isna().sum()}")
print(f"Zero heights: {(players_df['height'] == 0).sum()}")
print(f"Zero weights: {(players_df['weight'] == 0).sum()}")
print(f"Invalid birthdates: {(players_df['birthDate'] == '0000-00-00').sum()}")

# Assign positions based on distribution

# Get valid positions and their probabilities
valid_positions = players_df[players_df['pos'].notna()]['pos']
position_probs = valid_positions.value_counts(normalize=True)

# Assign random positions based on distribution to empty positions
empty_pos_mask = players_df['pos'].isna()
num_empty_pos = empty_pos_mask.sum()
if num_empty_pos > 0:
    np.random.seed(42)  # For reproducibility
    random_positions = np.random.choice(
        position_probs.index, 
        size=num_empty_pos, 
        p=position_probs.values
    )
    players_df.loc[empty_pos_mask, 'pos'] = random_positions
    print(f"\n✓ Assigned positions to {num_empty_pos} players")

# Calculate mean height and weight per position
position_stats = players_df[players_df['height'] > 0].groupby('pos').agg({
    'height': 'mean',
    'weight': 'mean'
}).round(1)

print("\nAverage stats by position:")
display(position_stats)

# Fill zero heights and weights with position means
for pos in position_stats.index:
    # Fill height
    height_mask = (players_df['pos'] == pos) & (players_df['height'] == 0)
    if height_mask.any():
        players_df.loc[height_mask, 'height'] = position_stats.loc[pos, 'height']
    
    # Fill weight
    weight_mask = (players_df['pos'] == pos) & (players_df['weight'] == 0)
    if weight_mask.any():
        players_df.loc[weight_mask, 'weight'] = position_stats.loc[pos, 'weight']

print(f"\n✓ Filled heights and weights based on position averages")

# Convert birth dates to NaT (Not a Time)
players_df['birthDate'] = players_df['birthDate'].replace('0000-00-00', pd.NaT)
print(f"\n✓ Converted invalid birthdates to NaT: {players_df['birthDate'].isna().sum()} missing values")

# Verify the changes
print("\n=== After Cleaning ===")
print(f"Empty positions: {(players_df['pos'] == '').sum()}")
print(f"Zero heights: {(players_df['height'] == 0).sum()}")
print(f"Zero weights: {(players_df['weight'] == 0).sum()}")
print(f"Invalid birthdates: {players_df['birthDate'].isna().sum()}")

print("\nNew position distribution:")
display(players_df['pos'].value_counts())

In [ ]:
# Filling missing values with "NA" -> good for scikit-learn
teams_df[["firstRound", "semis", "finals"]] = (
    teams_df[["firstRound", "semis", "finals"]]
    .fillna("NA")
)

### Cleaning - Outliers

In [ ]:
import scipy.stats

# Z-Score for detecting outliers 
z_scores = np.abs(scipy.stats.zscore(teams_df.select_dtypes(include=['number']))) 
outliers = (z_scores > 3).any(axis=1) 
teams_df_outliers = teams_df[outliers]

print(f"Number of outliers detected: {outliers.sum()}")
display(teams_df_outliers)

### Feature Engineering

In [ ]:
# Total wins and losses (regular season + playoff)
coaches_df["total_w"] = coaches_df["won"] + coaches_df["post_wins"]
coaches_df["total_l"] = coaches_df["lost"] + coaches_df["post_losses"]

In [ ]:
# per-game averages for key stats
players_df['ppg'] = players_teams_df['points'] / players_teams_df['GP']
players_df['apg'] = players_teams_df['assists'] / players_teams_df['GP']
players_df['rpg'] = players_teams_df['rebounds'] / players_teams_df['GP']
players_df['spg'] = players_teams_df['steals'] / players_teams_df['GP']
players_df['bpg'] = players_teams_df['blocks'] / players_teams_df['GP']
players_df['fg_pct'] = players_teams_df['fgMade'] / players_teams_df['fgAttempted']

In [ ]:
# Teams' win percentage
teams_df['win_pct'] = (teams_df['won'] / teams_df['GP']) * 100

In [ ]:
# Assist to turnover Ratio
players_teams_df["ast_to"] = (players_teams_df["assists"] + players_teams_df["PostAssists"]) / (players_teams_df["turnovers"] + players_teams_df["PostTurnovers"])

In [ ]:
## Player Efficiency Rating (PER)

players_teams_df["NBA_PER"] = (players_teams_df['points'] + players_teams_df['rebounds']+ players_teams_df['assists'] + players_teams_df['steals'] + players_teams_df['blocks']
  - ((players_teams_df['fgAttempted'] - players_teams_df['fgMade']) + (players_teams_df['ftAttempted'] - players_teams_df['ftMade']) + players_teams_df['turnovers'])
) / players_teams_df["GP"]


In [ ]:
# True Shooting Percentage - how efficiently a player converts their scoring opportunities
players_teams_df["ts_pct"] = players_teams_df['points'] / (2 * (players_teams_df["fgAttempted"] + (0.44 * players_teams_df["ftAttempted"])))

### Data Transformation

In [ ]:
# Teams Stats Histograms
teams_numeric_cols = teams_df.select_dtypes(include=['float64', 'int64']).filter(regex=r'^(d_|o_)')

teams_numeric_cols.describe()

In [ ]:
plt.figure(figsize=(16, 12))

for i, col in enumerate(teams_numeric_cols):
    plt.subplot(6, 6, i+1)
    sns.histplot(teams_numeric_cols[col], bins=30, kde=True, color='skyblue')
    plt.title(col)
    plt.tight_layout()

In [ ]:
# log transform
teams_numeric_cols_log = np.log(teams_df[teams_numeric_cols.columns])

In [ ]:
plt.figure(figsize=(16, 12))

for i, col in enumerate(teams_numeric_cols_log):
    plt.subplot(6, 6, i+1)
    sns.histplot(teams_numeric_cols_log[col], bins=30, kde=True, color='skyblue')
    plt.title(col)
    plt.tight_layout()

In [ ]:
# Convert Y/N to 0/1
teams_df['playoff'] = teams_df.playoff.map(dict(Y=1, N=0))

In [ ]:
# Isolation Forest
#iso = IsolationForest(contamination=0.01, random_state=42) # random state para ter smp mm res
#y_pred = iso.fit_predict(teams_df.select_dtypes(include=['number']))
#teams_df_outliers = teams_df[y_pred == -1]

#display(teams_df_outliers)